RQ1 ------------------------------------------------

XGBoost

Facemask wearing - Before mandates - covid model

Dataset with state and covid rolling cases and deaths are considered here for the analysis.

In [8]:
# import libraries
import pandas as pd
import numpy as np

In [9]:
before_train_facemask = pd.read_csv("before_train_facemask.csv")
before_test_facemask = pd.read_csv("before_test_facemask.csv")

In [10]:
before_train_facemask.columns

Index(['RecordNo', 'Date', 'Non-household contacts', 'age', 'household_size',
       'Wellbeing', 'Perceived Severity', 'Perceived Susceptibility',
       'face_mask_scale', 'face_mask_binary',
       'general_protective_behavior_scale',
       'general_protective_behavior_binary',
       'protective_behavior_nomask_scale', 'week', '7days_rolling_cases',
       '7days_rolling_deaths', 'mandate_start_date', 'mandate_period',
       'Isolate if unwell_Not sure', 'Isolate if unwell_Yes',
       'Isolate if instructed_Not sure',
       'Isolate if instructed_Somewhat unwilling',
       'Isolate if instructed_Somewhat willing',
       'Isolate if instructed_Very unwilling',
       'Isolate if instructed_Very willing', 'gender_Male',
       'state_New South Wales', 'state_Northern Territory', 'state_Queensland',
       'state_South Australia', 'state_Tasmania', 'state_Victoria',
       'state_Western Australia', 'employment_status_Not working',
       'employment_status_Part time employment'

In [11]:
# remove the target variables and identifiers
drop_cols = ['RecordNo', 'Date','face_mask_scale', 'face_mask_binary','general_protective_behavior_scale',
              'general_protective_behavior_binary','mandate_start_date']



# predictors
x_train = before_train_facemask.drop(columns=drop_cols)
x_test = before_test_facemask.drop(columns=drop_cols)

x_train = x_train.astype(float)  # converting boolean to float 
x_test = x_test.astype(float)

In [12]:
print(x_train.columns)
print(x_train.shape)

Index(['Non-household contacts', 'age', 'household_size', 'Wellbeing',
       'Perceived Severity', 'Perceived Susceptibility',
       'protective_behavior_nomask_scale', 'week', '7days_rolling_cases',
       '7days_rolling_deaths', 'mandate_period', 'Isolate if unwell_Not sure',
       'Isolate if unwell_Yes', 'Isolate if instructed_Not sure',
       'Isolate if instructed_Somewhat unwilling',
       'Isolate if instructed_Somewhat willing',
       'Isolate if instructed_Very unwilling',
       'Isolate if instructed_Very willing', 'gender_Male',
       'state_New South Wales', 'state_Northern Territory', 'state_Queensland',
       'state_South Australia', 'state_Tasmania', 'state_Victoria',
       'state_Western Australia', 'employment_status_Not working',
       'employment_status_Part time employment', 'employment_status_Retired',
       'employment_status_Unemployed',
       'Confidence in Goverment's response_A lot of confidence',
       'Confidence in Goverment's response_Don't 

In [13]:
# target variable 

y_train = before_train_facemask["face_mask_binary"]
y_test = before_test_facemask["face_mask_binary"]

XGBoost

In [14]:
# import libraries
from sklearn.model_selection import StratifiedKFold  
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
import joblib


cv = StratifiedKFold( # 5 fold cross validation
    n_splits=5,
    shuffle=True,
    random_state=42
)

  
cv_xgb = Pipeline([('ros', RandomOverSampler(random_state=42)),
                   ('xgb', XGBClassifier(random_state=42,
                                         eval_metric='logloss',
                                         n_jobs=-1))
    ])  

params = {
        'xgb__n_estimators': [100, 250],
        'xgb__learning_rate': [0.05, 0.1],  # 0.3
        #'xgb__subsample': [0.8,1.0],
        #'xgb__colsample_bytree': [0.8,1.0],
        'xgb__min_child_weight': [1, 5, 7],       
        #'xgb__gamma': [0, 0.1],          
        'xgb__max_depth': [3,5,7]
    }


grid = GridSearchCV(
        cv_xgb,
        params,
        cv=cv,
        scoring={  # evaluates all metrics
        'roc_auc': 'roc_auc',
        'accuracy': 'accuracy',
        'precision': 'precision',
        'recall': 'recall',
        'f1': 'f1'},
        refit='roc_auc',  # choose the best model
        return_train_score=False,

        # scoring='roc_auc', # for each combination, it uses 5 fold cv and calculates roc auc

        n_jobs=-1 # use all available CPU cores for parallel processing
    )

grid.fit(x_train, y_train)

best_xgb = grid.best_estimator_  # best model

best_parameters = grid.best_params_   # best hyper parameters

results_xgb = pd.DataFrame(grid.cv_results_)

pred = best_xgb.predict(x_test)
prob = best_xgb.predict_proba(x_test)[:,1]


print("Accuracy:", round(accuracy_score(y_test, pred),4))
print("ROC AUC:", round(roc_auc_score(y_test, prob),4))
print("Precision:", round(precision_score(y_test, pred),4))
print("Recall:", round(recall_score(y_test, pred),4))
print("F1:", round(f1_score(y_test, pred),4))


joblib.dump(best_xgb, "facemask_before_covidModel_XGBoost_rolling.pkl")

joblib.dump(best_parameters, "facemask_before_covidModel_XGBoost_bestParameters_rolling.pkl")

results_xgb.to_csv("facemask_before_covidModel_XGBoost_rolling_results.csv", index=False)

print(f"\nBest Parameters", grid.best_params_)

print(f"\nBest CV ROC AUC:", round(grid.best_score_,4))

Accuracy: 0.7713
ROC AUC: 0.8428
Precision: 0.5442
Recall: 0.6915
F1: 0.6091

Best Parameters {'xgb__learning_rate': 0.05, 'xgb__max_depth': 7, 'xgb__min_child_weight': 7, 'xgb__n_estimators': 250}

Best CV ROC AUC: 0.8565
